In [ ]:
import numpy as np
import cv2
import torch.nn.functional as F

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        self.hooks = [
            target_layer.register_forward_hook(self.save_activation),
            target_layer.register_full_backward_hook(self.save_gradient)
        ]

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate_heatmap(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        output = self.model(input_tensor)
        
        if class_idx is None:
            class_idx = torch.argmax(output, dim=1).item()
        
        # Backward pass for the specific class
        loss = output[0, class_idx]
        loss.backward()

        # Weight the filters by the gradients (Global Average Pooling)
        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        
        # Weighted sum of feature maps
        cam = torch.sum(weights * self.activations, dim=1).squeeze()
        
        # ReLU to keep only features that have a positive influence on the class
        cam = np.maximum(cam.detach().cpu().numpy(), 0)
        
        # Normalize and Resize
        cam = cv2.resize(cam, (input_tensor.shape[2], input_tensor.shape[3]))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

    def __del__(self):
        for hook in self.hooks:
            hook.remove()

In [ ]:
def apply_colormap_on_image(org_img, heatmap):
    # Convert heatmap to RGB colormap
    heatmap_img = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap_img = cv2.cvtColor(heatmap_img, cv2.COLOR_BGR2RGB)
    
    # Superimpose
    result = heatmap_img * 0.4 + org_img * 0.6
    return np.uint8(result)

In [ ]:
# Assuming 'model' is your ResNet18
target_layer = model.layer4[-1] 
cam_extractor = GradCAM(model, target_layer)

In [ ]:
# In torchvision EfficientNet, features are in a Sequential block
target_layer = model.features[-1] 
cam_extractor = GradCAM(model, target_layer)

In [ ]:
model.eval()
input_image, label = next(iter(val_loader))
input_tensor = input_image[0].unsqueeze(0).to(device)

# Generate Heatmap
heatmap, pred_idx = cam_extractor.generate_heatmap(input_tensor)

# Prepare original image for display (Undo normalization)
img_np = input_image[0].permute(1, 2, 0).cpu().numpy()
img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
img_np = np.clip(img_np, 0, 1) * 255

# Visualize
final_viz = apply_colormap_on_image(img_np, heatmap)
plt.imshow(final_viz)
plt.title(f"Pred: {pred_idx} | Actual: {label[0].item()}")
plt.show()